# 🔬 SpecWatson: Simulación y Ajuste de Espectros Electrónicos de Alta Resolución
### **Proyecto de Tesis:** Desarrollo de Software para Espectroscopía Rovibrónica Molecular
**Estudiante:** Miguel Díaz  
**Director:** Dr. Leonardo Álvarez Valtierra  
**Institución:** Laboratorio de Espectroscopía Molecular, División de Ciencias e Ingenierías, Universidad de Guanajuato  

---

### 📌 Objetivos de este Cuaderno en Google Colab:
1. **Verificación de Entorno y GPU:** Detección automática de aceleración por hardware.
2. **Modelo Físico:** Implementación del Hamiltoniano de Rotor Rígido y Hamiltoniano de Watson (reducción tipo A en representación $I^r$) para rotores asimétricos.
3. **Simulación Espectral:** Cálculo de transiciones rovibrónicas $S_1 \leftarrow S_0$ permitidas por dipolo eléctrico (bandas híbridas $a/b/c$) en condiciones de jet supersónico ($T \sim 3-5\text{ K}$).
4. **Convolución de Perfiles:** Generación de contornos espectrales continuos (Gauss/Voigt) acordes al ancho de banda del láser experimental (FWHM $\approx 30\text{ MHz}$).
5. **Ajuste Automatizado (Problema Inverso):** Ajuste de contorno de banda por Algoritmo Genético / Evolución Diferencial sin necesidad de asignación manual de líneas.

In [ ]:
# @title 1. Configuración del Entorno y Dependencias en Colab
# Ejecuta esta celda para verificar GPU e instalar paquetes científicos

!nvidia-smi

import sys
import numpy as np
import scipy
from scipy.special import factorial, voigt_profile
from scipy.optimize import differential_evolution, minimize
import matplotlib.pyplot as plt

print(f"✓ NumPy versión: {np.__version__}")
print(f"✓ SciPy versión: {scipy.__version__}")

try:
    import torch
    gpu_avail = torch.cuda.is_available()
    print(f"✓ PyTorch versión: {torch.__version__} | GPU Disponible: {gpu_avail}")
    if gpu_avail:
        print(f"   Dispositivo GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("! PyTorch no detectado (usando NumPy/SciPy de alto rendimiento en CPU)")

## 2. Definición del Motor Cuántico: Hamiltoniano de Watson y Rotor Asimétrico
Para un rotor asimétrico en la representación $I^r$ ($z=a, x=b, y=c$):
$$H_0 = \frac{1}{2}(B+C)\hat{J}^2 + \left[A - \frac{1}{2}(B+C)\right]\hat{J}_a^2 + \frac{1}{4}(B-C)(\hat{J}_+^2 + \hat{J}_-^2)$$
Añadiendo los términos de distorsión centrífuga de Watson (Reducción A):
$$H_W = H_0 - \Delta_J \hat{J}^4 - \Delta_{JK} \hat{J}^2 \hat{J}_a^2 - \Delta_K \hat{J}_a^4 - 2\delta_J \hat{J}^2(\hat{J}_b^2 - \hat{J}_c^2) - \delta_K [\hat{J}_a^2, (\hat{J}_b^2 - \hat{J}_c^2)]_+$$

In [ ]:
# @title 2. Módulo del Hamiltoniano y Diagonalización

class MoleculeState:
    def __init__(self, name="State", A=3000.0, B=1500.0, C=1000.0,
                 Delta_J=0.0, Delta_JK=0.0, Delta_K=0.0, delta_J=0.0, delta_K=0.0, origin=0.0):
        self.name = name
        self.A = float(A)  # Constantes en MHz
        self.B = float(B)
        self.C = float(C)
        self.Delta_J = float(Delta_J)
        self.Delta_JK = float(Delta_JK)
        self.Delta_K = float(Delta_K)
        self.delta_J = float(delta_J)
        self.delta_K = float(delta_K)
        self.origin = float(origin) # Origen de banda en cm^-1

    @property
    def kappa(self):
        """Parámetro de asimetría de Ray kappa = (2B - A - C) / (A - C)"""
        if abs(self.A - self.C) < 1e-9:
            return 0.0
        return (2.0 * self.B - self.A - self.C) / (self.A - self.C)


class HamiltonianEngine:
    def __init__(self, state: MoleculeState):
        self.state = state

    def build_matrix(self, J: int) -> np.ndarray:
        dim = 2 * J + 1
        H = np.zeros((dim, dim), dtype=np.float64)
        K_vals = np.arange(-J, J + 1)
        A, B, C = self.state.A, self.state.B, self.state.C
        dJ, dJK, dK = self.state.Delta_J, self.state.Delta_JK, self.state.Delta_K
        del_j, del_k = self.state.delta_J, self.state.delta_K
        JJ1 = J * (J + 1)

        # Elementos diagonales <J, K | H | J, K>
        for i, K in enumerate(K_vals):
            diag_rigid = 0.5 * (B + C) * JJ1 + (A - 0.5 * (B + C)) * (K ** 2)
            diag_watson = - dJ * (JJ1 ** 2) - dJK * JJ1 * (K ** 2) - dK * (K ** 4)
            H[i, i] = diag_rigid + diag_watson

        # Elementos fuera de la diagonal <J, K+2 | H | J, K>
        for i, K in enumerate(K_vals):
            if K + 2 <= J:
                j_idx = i + 2
                term1 = JJ1 - K * (K + 1)
                term2 = JJ1 - (K + 1) * (K + 2)
                if term1 > 0 and term2 > 0:
                    f_fac = np.sqrt(term1 * term2)
                    off_rigid = 0.25 * (B - C) * f_fac
                    off_watson = - (2.0 * del_j * JJ1 + del_k * ((K + 2)**2 + K**2)) * 0.5 * f_fac
                    val = off_rigid + off_watson
                    H[i, j_idx] = val
                    H[j_idx, i] = val
        return H

    def solve_j(self, J: int):
        H = self.build_matrix(J)
        energies, vectors = np.linalg.eigh(H)
        # Asignación de etiquetas (Ka, Kc)
        labels = []
        for i in range(2 * J + 1):
            tau = -J + i
            Ka_plus_Kc = J if (J - tau) % 2 == 0 else J + 1
            Ka = (Ka_plus_Kc + tau) // 2
            Kc = (Ka_plus_Kc - tau) // 2
            labels.append((int(Ka), int(Kc)))
        return energies, vectors, labels

    def compute_manifold(self, J_max: int):
        manifold = {}
        for J in range(J_max + 1):
            energies, vectors, labels = self.solve_j(J)
            manifold[J] = {
                "energies": energies,
                "vectors": vectors,
                "labels": labels,
                "K_vals": np.arange(-J, J + 1)
            }
        return manifold

print("✓ Motor de Hamiltoniano cargado correctamente.")

## 3. Transiciones Rovibrónicas y Convolución Instrumental
Cálculo de reglas de selección, poblaciones térmicas de Boltzmann y convolución de perfiles de Voigt.

In [ ]:
# @title 3. Módulo de Transiciones e Intensidades

def wigner_3j(j1, j2, j3, m1, m2, m3):
    if m1 + m2 + m3 != 0 or not (abs(j1 - j2) <= j3 <= j1 + j2) or abs(m1) > j1 or abs(m2) > j2 or abs(m3) > j3:
        return 0.0
    t_min = max(0, j2 - j3 - m1, j1 - j3 + m2)
    t_max = min(j1 + j2 - j3, j1 - m1, j2 + m2)
    if t_min > t_max:
        return 0.0
    sum_val = sum(
        ((-1.0) ** t) / (
            factorial(t) * factorial(j1 + j2 - j3 - t) * factorial(j1 - m1 - t) *
            factorial(j2 + m2 - t) * factorial(j3 - j2 + m1 + t) * factorial(j3 - j1 - m2 + t)
        ) for t in range(int(t_min), int(t_max) + 1)
    )
    prefactor = ((-1.0) ** (j1 - j2 - m3)) * np.sqrt(
        (factorial(j1 + j2 - j3) * factorial(j1 - j2 + j3) * factorial(-j1 + j2 + j3)) / factorial(j1 + j2 + j3 + 1)
    ) * np.sqrt(
        factorial(j1 + m1) * factorial(j1 - m1) * factorial(j2 + m2) * factorial(j2 - m2) * factorial(j3 + m3) * factorial(j3 - m3)
    )
    return prefactor * sum_val

KB_MHZ = 20836.61912
WAVENUMBER_TO_MHZ = 29979.2458

def simulate_rovibronic_spectrum(g_manifold, e_manifold, band_origin_cm1=30000.0, T_kelvin=4.0,
                                 mu_a=1.0, mu_b=0.0, mu_c=0.0, J_max=15, fwhm_mhz=30.0, grid_points=4000):
    # Función de partición Q_rot
    E0 = g_manifold[0]["energies"][0]
    Q = sum((2*J + 1) * np.sum(np.exp(-(g_manifold[J]["energies"] - E0) / (KB_MHZ * T_kelvin))) for J in g_manifold)

    lines_freq = []
    lines_int = []
    band_origin_mhz = band_origin_cm1 * WAVENUMBER_TO_MHZ

    for J_low in range(J_max + 1):
        g_d = g_manifold[J_low]
        for J_up in [J_low - 1, J_low, J_low + 1]:
            if J_up < 0 or J_up > J_max or (J_low == 0 and J_up == 0):
                continue
            e_d = e_manifold[J_up]
            for i_l, (Ka_l, Kc_l) in enumerate(g_d["labels"]):
                E_low = g_d["energies"][i_l]
                pop = (2 * J_low + 1) * np.exp(-(E_low - E0) / (KB_MHZ * T_kelvin)) / Q
                if pop < 1e-10: continue
                v_l = g_d["vectors"][:, i_l]

                for i_u, (Ka_u, Kc_u) in enumerate(e_d["labels"]):
                    E_up = e_d["energies"][i_u]
                    v_u = e_d["vectors"][:, i_u]
                    d_Ka, d_Kc = abs(Ka_u - Ka_l), abs(Kc_u - Kc_l)
                    is_a = (d_Ka % 2 == 0) and (d_Kc % 2 != 0) and (mu_a > 0)
                    is_b = (d_Ka % 2 != 0) and (d_Kc % 2 != 0) and (mu_b > 0)

                    if not (is_a or is_b): continue
                    S_a = 0.0
                    if is_a:
                        amp = 0.0
                        for kl_i, kl in enumerate(g_d["K_vals"]):
                            if kl in e_d["K_vals"]:
                                ku_i = np.where(e_d["K_vals"] == kl)[0][0]
                                amp += v_u[ku_i] * v_l[kl_i] * wigner_3j(J_up, 1, J_low, -kl, 0, kl)
                        S_a = (2 * J_up + 1) * (2 * J_low + 1) * (amp ** 2)
                    
                    intensity = pop * (mu_a * S_a)
                    if intensity > 1e-10:
                        freq = band_origin_mhz + (E_up - E_low)
                        lines_freq.append(freq)
                        lines_int.append(intensity)

    lines_freq = np.array(lines_freq)
    lines_int = np.array(lines_int)

    # Convolución Gaussiana / Voigt con FWHM
    f_min, f_max = np.min(lines_freq) - 150.0, np.max(lines_freq) + 150.0
    x_axis = np.linspace(f_min, f_max, grid_points)
    y_axis = np.zeros_like(x_axis)
    sigma = fwhm_mhz / (2.0 * np.sqrt(2.0 * np.log(2.0)))

    for f0, I0 in zip(lines_freq, lines_int):
        dx = x_axis - f0
        y_axis += I0 * np.exp(-0.5 * (dx / sigma) ** 2)

    y_norm = y_axis / np.max(y_axis)
    return x_axis, y_norm, lines_freq, lines_int

print("✓ Módulo de transiciones cargado.")

## 4. Experimento de Simulación: Molécula Modelo en Expansión Supersónica
Simularemos el espectro $S_1 \leftarrow S_0$ de una molécula tipo hidrocarburo aromático / derivado de anisol (inspirado en los sistemas estudiados por el grupo del Dr. Álvarez Valtierra):
* **Estado Basal ($S_0$):** $A'' = 2178.6\text{ MHz}, B'' = 1251.2\text{ MHz}, C'' = 794.5\text{ MHz}$
* **Estado Excitado ($S_1$):** $A' = 2145.3\text{ MHz}, B' = 1242.1\text{ MHz}, C' = 788.9\text{ MHz}$
* **Temperatura rotacional en el jet:** $T_{rot} = 4.0\text{ K}$

In [ ]:
# @title 4. Ejecución de la Simulación y Visualización Espectral

ground_state = MoleculeState(
    name="S0_Ground",
    A=2178.60, B=1251.20, C=794.50,
    Delta_J=0.0002, Delta_JK=-0.0005, Delta_K=0.001,
    origin=0.0
)

excited_state = MoleculeState(
    name="S1_Excited",
    A=2145.30, B=1242.10, C=788.90,
    Delta_J=0.0002, Delta_JK=-0.0005, Delta_K=0.001,
    origin=30000.0 # cm^-1
)

print(f"Parámetro de asimetría kappa (S0): {ground_state.kappa:.4f}")
print(f"Parámetro de asimetría kappa (S1): {excited_state.kappa:.4f}")

# Resolver niveles de energía
J_MAX = 12
g_ham = HamiltonianEngine(ground_state)
e_ham = HamiltonianEngine(excited_state)

g_manifold = g_ham.compute_manifold(J_MAX)
e_manifold = e_ham.compute_manifold(J_MAX)

# Simular espectro con ancho de banda láser FWHM = 30 MHz
x_freq, y_cont, st_freq, st_int = simulate_rovibronic_spectrum(
    g_manifold, e_manifold,
    band_origin_cm1=30000.0,
    T_kelvin=3.5,
    mu_a=1.0,
    J_max=J_MAX,
    fwhm_mhz=30.0,
    grid_points=3500
)

# Frecuencia relativa al origen en MHz
origin_mhz = 30000.0 * WAVENUMBER_TO_MHZ
rel_freq_mhz = x_freq - origin_mhz
rel_stick_mhz = st_freq - origin_mhz

# Gráfica de Publicación
plt.figure(figsize=(12, 5.5), dpi=140)
plt.plot(rel_freq_mhz, y_cont, color="#1a5276", lw=1.6, label="Contorno Convolucionado (Voigt, FWHM=30 MHz)")
plt.vlines(rel_stick_mhz, 0, st_int / np.max(st_int), color="#e74c3c", alpha=0.5, lw=1.1, label="Transiciones Individuales (Stick Spectrum)")

plt.title(r"Espectro Electrónico de Alta Resolución $S_1 \leftarrow S_0$ (Resolución Rotacional)", fontsize=13, weight="bold")
plt.xlabel("Frecuencia Relativa $\\nu - \\nu_0$ (MHz)", fontsize=11)
plt.ylabel("Intensidad Normalizada", fontsize=11)
plt.axvline(0, color="black", linestyle="--", alpha=0.4, label="Origen de Banda $\\nu_0$")
plt.legend(frameon=True, facecolor="white", loc="upper right")
plt.grid(True, alpha=0.25, linestyle=":")
plt.tight_layout()
plt.show()

## 5. El Problema Inverso: Ajuste Automatizado por Algoritmo Genético
Supongamos que el espectro simulado anterior es nuestro **"espectro experimental"** obtenido en el laboratorio láser de haces moleculares.

A continuación, asumimos que **no conocemos las constantes exactas del estado excitado** y le damos al algoritmo un rango amplio de búsqueda para que las determine automáticamente mediante optimización global.

In [ ]:
# @title 5. Ajuste de Contorno Automatizado (Differential Evolution)

# Generar datos "experimentales" con ruido gaussiano del 3%
np.random.seed(42)
exp_x = rel_freq_mhz
exp_y = y_cont + np.random.normal(0, 0.03, size=len(y_cont))
exp_y = np.clip(exp_y, 0, None)
exp_y = exp_y / np.max(exp_y)

print("Datos experimentales preparados con ruido instrumental.")

# Función objetivo: correlación cruzada normalizada entre espectro simulado y experimental
def fitness_objective(params):
    A_cand, B_cand, C_cand = params
    if not (A_cand >= B_cand >= C_cand > 0):
        return 1.0
    
    # Calcular estado excitado candidato
    cand_state = MoleculeState("Cand", A=A_cand, B=B_cand, C=C_cand, origin=30000.0)
    cand_ham = HamiltonianEngine(cand_state)
    cand_manifold = cand_ham.compute_manifold(8) # J_max reducido para ajuste ultra-rápido
    
    _, sim_y, _, _ = simulate_rovibronic_spectrum(
        g_manifold, cand_manifold,
        band_origin_cm1=30000.0, T_kelvin=3.5, mu_a=1.0, J_max=8, fwhm_mhz=30.0, grid_points=len(exp_x)
    )
    
    # Correlación normalizada
    dot = np.dot(exp_y, sim_y)
    norm = np.linalg.norm(exp_y) * np.linalg.norm(sim_y)
    if norm < 1e-9:
        return 1.0
    return - (dot / norm)

# Límites de búsqueda para A', B', C' en MHz
bounds = [
    (2100.0, 2200.0), # Rango para A'
    (1200.0, 1280.0), # Rango para B'
    (750.0, 820.0)    # Rango para C'
]

print("Iniciando optimización por Algoritmo Genético (Differential Evolution)... Espere unos segundos...")
fit_res = differential_evolution(fitness_objective, bounds, maxiter=12, popsize=8, seed=42, disp=True)

A_fit, B_fit, C_fit = fit_res.x
print("\n=================== RESULTADOS DEL AJUSTE ===================")
print(f"A' Real: {excited_state.A:.2f} MHz | A' Ajustado: {A_fit:.2f} MHz | Error: {abs(A_fit - excited_state.A):.2f} MHz")
print(f"B' Real: {excited_state.B:.2f} MHz | B' Ajustado: {B_fit:.2f} MHz | Error: {abs(B_fit - excited_state.B):.2f} MHz")
print(f"C' Real: {excited_state.C:.2f} MHz | C' Ajustado: {C_fit:.2f} MHz | Error: {abs(C_fit - excited_state.C):.2f} MHz")
print(f"Correlación espectral alcanzada: {-fit_res.fun * 100:.2f}%")
print("=============================================================")

## 6. Comparación Final: Espectro Experimental vs Ajustado y Residuales
Visualización directa del ajuste y del residuo $(I_{exp} - I_{calc})$ típico de reportes de tesis y publicaciones arbitradas.

In [ ]:
# @title 6. Gráfica de Residuos y Bondad de Ajuste

# Generar espectro final con las constantes ajustadas
final_state = MoleculeState("Final_Fit", A=A_fit, B=B_fit, C=C_fit, origin=30000.0)
final_ham = HamiltonianEngine(final_state)
final_manifold = final_ham.compute_manifold(J_MAX)

_, final_y, _, _ = simulate_rovibronic_spectrum(
    g_manifold, final_manifold,
    band_origin_cm1=30000.0, T_kelvin=3.5, mu_a=1.0, J_max=J_MAX, fwhm_mhz=30.0, grid_points=len(exp_x)
)

residual = exp_y - final_y

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={'height_ratios': [3, 1]}, dpi=140)

# Espectro Superior
ax1.plot(exp_x, exp_y, color="black", lw=1.2, label="Espectro Experimental (Con Ruido)")
ax1.plot(exp_x, final_y, color="#e74c3c", lw=1.4, linestyle="--", label=f"Ajuste Automatizado (A'={A_fit:.1f}, B'={B_fit:.1f}, C'={C_fit:.1f} MHz)")
ax1.set_title("Ajuste de Espectro Electrónico de Alta Resolución por Algoritmo Genético", fontsize=13, weight="bold")
ax1.set_ylabel("Intensidad Relativa", fontsize=11)
ax1.legend(loc="upper right", frameon=True)
ax1.grid(True, alpha=0.25, linestyle=":")

# Residuales Inferiores
ax2.plot(exp_x, residual, color="#27ae60", lw=1.0, label="Residuo (Exp - Calc)")
ax2.axhline(0, color="black", lw=0.8, linestyle="-")
ax2.set_xlabel("Frecuencia Relativa $\\nu - \\nu_0$ (MHz)", fontsize=11)
ax2.set_ylabel("Residuo", fontsize=11)
ax2.set_ylim(-0.25, 0.25)
ax2.legend(loc="upper right", frameon=True)
ax2.grid(True, alpha=0.25, linestyle=":")

plt.tight_layout()
plt.show()